In [1]:
import pandas as pd
import numpy as np

In [42]:
# --- 1. SETUP ----
MAIN_DATA_FILE = 'C:/Users/raffi/OneDrive - United Nations/Desktop/DSS/DATA AVAILABILITY INTERACTIVE DASHBOARD/final_dataset_combined.xlsx'
CRITERIA_FILE = 'C:/Users/raffi/OneDrive - United Nations/Desktop/DSS/DATA AVAILABILITY INTERACTIVE DASHBOARD/criterias.xlsx'

# Load data
main_df = pd.read_excel(MAIN_DATA_FILE)
criteria_df = pd.read_excel(CRITERIA_FILE).dropna(subset=['Indicator_Ar'])[['Indicator_Ar', 'criteria']]

# Force numeric values (Invalid text becomes NaN)
main_df['val_numeric'] = pd.to_numeric(main_df['العدد'], errors='coerce')
# Convert Year to numeric, turning any non-year text into NaN
main_df['السنة'] = pd.to_numeric(main_df['السنة'], errors='coerce')

In [56]:
# --- 2. GENERAL AVAILABILITY ---

# 2. Define conditions based on the CURRENT dataframe being processed
# Move these inside the logic so they apply to main_df before it's grouped
conds = [
    (main_df['السنة'] >= 2010) & (main_df['السنة'] <= 2014),
    (main_df['السنة'] >= 2015) & (main_df['السنة'] <= 2019),
    (main_df['السنة'] >= 2020) & (main_df['السنة'] <= 2026)
]
bins = ['2010-2014', '2015-2019', '2020-2026']

# 3. Add the bracket to the full dataframe FIRST
main_df['bracket'] = np.select(conds, bins, default='Other')

# 4. Now perform the collapse. Include 'bracket' in the groupby so you don't lose it.
df_gen = main_df[main_df['bracket'] != 'Other'].copy()
df_gen = df_gen.groupby(['المؤشر', 'الدولة', 'السنة', 'bracket'])['val_numeric'].max().reset_index()

# 5. Add valid data flag
df_gen['is_valid'] = df_gen['val_numeric'].notna().astype(int)

# 6. Sum valid years per bracket
gen_brk = df_gen.groupby(['المؤشر', 'الدولة', 'bracket'])['is_valid'].sum().reset_index()

# 7. Calculate Group Size
gen_brk['group_size'] = gen_brk.groupby(['المؤشر', 'الدولة'])['bracket'].transform('size')

# 8. Merge Criteria and Apply Logic
gen_final = pd.merge(gen_brk, criteria_df, left_on='المؤشر', right_on='Indicator_Ar', how='left')

# Availability = 1 if (group_size is 3) AND (every bracket count >= criteria)
gen_final['is_ok'] = ((gen_final['group_size'] == 3) & (gen_final['is_valid'] >= gen_final['criteria'])).astype(int)

# 9. Final Collapse
final_gen_report = gen_final.groupby(['المؤشر', 'الدولة'])['is_ok'].min().reset_index(name='general_availability')

In [ ]:
# --- 3. NATIONALITY AVAILABILITY ---

# Initial Clean 
# (Note: main_df already has the 'bracket' column from the General Availability step)
df_nat = main_df.dropna(subset=['المواطنة', 'val_numeric'])
df_nat = df_nat[df_nat['المواطنة'].isin(['مواطنون','غير مواطنين'])]

# This collapses Male and Female for the same year into one record.
# If either (or both) exists, this year is now "active" with one value.
df_sex_yearly = df_nat.groupby(['المؤشر', 'الدولة', 'bracket'])['val_numeric'].max().reset_index()

# Sum valid years per bracket (Use count() on the value column to ignore NaNs)
nat_brk = df_nat.groupby(['المؤشر', 'الدولة', 'bracket'])['val_numeric'].count().reset_index(name='valid_year_count')
# Get Group Size (Number of brackets)
nat_brk['group_size'] = nat_brk.groupby(['المؤشر', 'الدولة'])['bracket'].transform('count')

# Merge Criteria and Apply Logic
nat_final = pd.merge(nat_brk, criteria_df, left_on='المؤشر', right_on='Indicator_Ar', how='left')
nat_final['is_ok'] = ((nat_final['group_size'] == 3) & (nat_final['valid_year_count'] >= nat_final['criteria'])).astype(int)

# Final Collapse
final_nat_report = nat_final.groupby(['المؤشر', 'الدولة'])['is_ok'].min().reset_index(name='nationality_availability')

In [58]:
nat_brk

,المؤشر,الدولة,bracket,valid_year_count,group_size
0,استخدام وسائل منع الحمل (الحديثة والتقليدية) (...,ليبيا,2010-2014,1,2
1,استخدام وسائل منع الحمل (الحديثة والتقليدية) (...,ليبيا,2020-2026,1,2
2,استخدام وسائل منع الحمل التقليدية (نسبة مئوية)...,ليبيا,2010-2014,1,2
3,استخدام وسائل منع الحمل التقليدية (نسبة مئوية)...,ليبيا,2020-2026,1,2
4,استخدام وسائل منع الحمل الحديثة (نسبة مئوية) ح...,ليبيا,2010-2014,1,2
5,استخدام وسائل منع الحمل الحديثة (نسبة مئوية) ح...,ليبيا,2020-2026,1,2
6,الأفراد خارج القوى العاملة حسب الفئة العمرية,العراق,2020-2026,1,1
7,الرعاية قبل الولادة (4 زيارات على الأقل أثناء ...,ليبيا,2010-2014,1,2
8,الرعاية قبل الولادة (4 زيارات على الأقل أثناء ...,ليبيا,2020-2026,1,2
9,الولادات التي يشرف عليها أخصائيون صحّيون مَهَر...,ليبيا,2010-2014,1,2


In [59]:
any(final_nat_report['nationality_availability']==1)

False

In [ ]:
# --- 4. SEX AVAILABILITY (Mirroring Nationality Logic) ---

# Initial Clean
df_sex = main_df.dropna(subset=['الجنس', 'val_numeric'])
df_sex = df_sex[df_sex['الجنس'].isin(['ذكور','إناث'])]

# This collapses Male and Female for the same year into one record.
# If either (or both) exists, this year is now "active" with one value.
df_sex_yearly = df_sex.groupby(['المؤشر', 'الدولة', 'bracket'])['val_numeric'].max().reset_index()

# Sum valid years per bracket (Use count() on the value column to ignore NaNs)
sex_brk = df_sex.groupby(['المؤشر', 'الدولة', 'bracket'])['val_numeric'].count().reset_index(name='valid_year_count')

# Get Group Size (Number of brackets)
sex_brk['group_size'] = sex_brk.groupby(['المؤشر', 'الدولة'])['bracket'].transform('count')

# Merge Criteria and Apply Logic
sex_final = pd.merge(sex_brk, criteria_df, left_on='المؤشر', right_on='Indicator_Ar', how='left')
sex_final['is_ok'] = ((sex_final['group_size'] == 3) & (sex_final['valid_year_count'] >= sex_final['criteria'])).astype(int)

# Final Collapse
final_sex_report = sex_final.groupby(['المؤشر', 'الدولة'])['is_ok'].min().reset_index(name='sex_availability')

In [ ]:
# --- 5. THE MASTER MERGER (WITH CHAPTER) ---

# A. Create a unique mapping of Indicators to Chapters
# This ensures every row in the final report knows which chapter it belongs to
indicator_chapter_map = main_df[['المؤشر', 'الفصل']].drop_duplicates()

# B. Start with final_gen_report and merge the Chapter mapping
# We do this first so 'الفصل' is at the beginning of the file
master_file = pd.merge(final_gen_report, indicator_chapter_map, on='المؤشر', how='left')

# C. Merge Nationality and Sex results
master_file = pd.merge(master_file, final_nat_report, on=['المؤشر', 'الدولة'], how='left')
master_file = pd.merge(master_file, final_sex_report, on=['المؤشر', 'الدولة'], how='left')

# D. Cleanup: Fill NaNs and ensure integers for Power BI
cols_to_fix = ['nationality_availability', 'sex_availability']
master_file[cols_to_fix] = master_file[cols_to_fix].fillna(0).astype(int)
master_file['general_availability'] = master_file['general_availability'].astype(int)

# E. Reorder columns for better readability in Excel/Power BI
# Moving Chapter to the third position
cols = ['المؤشر', 'الفصل', 'الدولة', 'general_availability', 'nationality_availability', 'sex_availability']
master_file = master_file[cols]

# Save Final Master File
master_file.to_excel('master_availability.xlsx', index=False)

print("Process Complete. Master combined file (with Chapters) saved.")

In [61]:
any(final_sex_report['sex_availability']==1)

True

### creating the tables for the PBI TAG report

In [63]:
# --- 5. POWER BI TABLE 1: MAIN GRANULAR ---
# Ensure numeric and handle grouping
main_df['val_numeric'] = pd.to_numeric(main_df['العدد'], errors='coerce')
# Convert Year to numeric, turning any non-year text into NaN
main_df['السنة'] = pd.to_numeric(main_df['السنة'], errors='coerce')

# Grouping by Country, Indicator, Chapter (الفصل), and Year
pbi_main = main_df.groupby(['الدولة', 'المؤشر', 'الفصل', 'السنة'])['val_numeric'].max().reset_index()

# Availability flag: 1 if it's a number, 0 if NaN
pbi_main['available'] = pbi_main['val_numeric'].notna().astype(int)

# --- 6. POWER BI TABLE 2: NATIONALITY & SEX GRANULAR ---

# Nationality
#use pbi_nat from above
pbi_nat = df_nat.groupby(['الدولة', 'المؤشر', 'الفصل', 'السنة'])['val_numeric'].max().reset_index()
pbi_nat['nat_available'] = pbi_nat['val_numeric'].notna().astype(int)

# Sex
#use pbi_sex form above
pbi_sex = df_sex.groupby(['الدولة', 'المؤشر', 'الفصل', 'السنة'])['val_numeric'].max().reset_index()
pbi_sex['sex_available'] = pbi_sex['val_numeric'].notna().astype(int)

In [64]:
# Merge them together
# Start with Nationality, merge Sex, then merge into Main to ensure all indicators are kept
granular_master = pd.merge(pbi_main, pbi_nat[['الدولة', 'المؤشر', 'السنة', 'nat_available']], 
                           on=['الدولة', 'المؤشر', 'السنة'], how='left')

granular_master = pd.merge(granular_master, pbi_sex[['الدولة', 'المؤشر', 'السنة', 'sex_available']], 
                           on=['الدولة', 'المؤشر', 'السنة'], how='left')

# Clean up the resulting file
# Fill NaNs with 0 (meaning no data for that specific sub-category in that year)
granular_master[['nat_available', 'sex_available']] = granular_master[['nat_available', 'sex_available']].fillna(0).astype(int)

# Save to a single file
granular_master.to_excel('pbi_granular.xlsx', index=False)

print("Unified granular master file saved with all availability flags.")

Unified granular master file saved with all availability flags.


In [65]:
# --- 7. SUMMARY PERCENTAGE TABLES (DIRECT LINEAR) ---

# A. Create a mapping of Indicators to Chapters and get Denominators
indicator_map = main_df[['المؤشر', 'الفصل']].drop_duplicates()
chapter_totals = indicator_map.groupby('الفصل')['المؤشر'].nunique().reset_index(name='total_in_chapter')

# B. PREPARE GENERAL (The Base for the merge)
report_gen_with_chapter = pd.merge(final_gen_report, indicator_map, on='المؤشر', how='left')
summary_gen = report_gen_with_chapter.groupby(['الدولة', 'الفصل'])['general_availability'].sum().reset_index()

# C. PREPARE NATIONALITY
report_nat_with_chapter = pd.merge(final_nat_report, indicator_map, on='المؤشر', how='left')
summary_nat = report_nat_with_chapter.groupby(['الدولة', 'الفصل'])['nationality_availability'].sum().reset_index()

# D. PREPARE SEX
report_sex_with_chapter = pd.merge(final_sex_report, indicator_map, on='المؤشر', how='left')
summary_sex = report_sex_with_chapter.groupby(['الدولة', 'الفصل'])['sex_availability'].sum().reset_index()

# --- E. THE UNIFIED SUMMARY MERGE ---

# 1. Merge Nationality into General
summary_master = pd.merge(summary_gen, summary_nat, on=['الدولة', 'الفصل'], how='left')

# 2. Merge Sex into the result
summary_master = pd.merge(summary_master, summary_sex, on=['الدولة', 'الفصل'], how='left')

# 3. Add Denominators and Calculate Percentages
summary_master = pd.merge(summary_master, chapter_totals, on='الفصل', how='left')

# Fill any NaNs from the merge with 0
summary_master[['nationality_availability', 'sex_availability']] = summary_master[['nationality_availability', 'sex_availability']].fillna(0)

# Calculate all 3 Percentages
summary_master['pct_general'] = (summary_master['general_availability'] / summary_master['total_in_chapter']) * 100
summary_master['pct_nationality'] = (summary_master['nationality_availability'] / summary_master['total_in_chapter']) * 100
summary_master['pct_sex'] = (summary_master['sex_availability'] / summary_master['total_in_chapter']) * 100

# Save the unified summary file
summary_master.to_excel('summary_pct_master_unified.xlsx', index=False)

print("Unified Summary Table saved with all percentage columns.")

Unified Summary Table saved with all percentage columns.


In [66]:
sex_availability

NameError: name 'sex_availability' is not defined